# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimanshahid800/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: Scoring / Ranking.

This is not plain classification because the real goal is not just to
label a page as "declining" or "not declining" — it is to produce a
ranked priority list, so a reviewer with limited time checks the most
urgent pages first. The output is an ordered queue, not a single label.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#no code needed !!

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target/proxy: is_declining_label = (trend_direction == "down")

This label comes from the current data window (an observed rule-based
bucket), not a true future outcome — so it is a proxy, not a perfect
label. A stronger version would look at the next 30 days after a
90-day feature window, but this starter proxy is good enough to learn
the workflow end to end.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#no code needed

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@50

This asks: of the top 50 pages my model ranks as highest priority,
how many are actually declining? This matches how the output is used
in real life — a reviewer only has time to check a limited number of
pages, so it matters that the top of the list is correct, not the
whole ranking.

"Good" means beating a simple fixed rule. The starter baseline scores
0.240 on this metric; a random forest model scores 0.740 — meaning
roughly 37 out of the top 50 pages are correctly identified, versus
only 12 with the baseline rule.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row = one content page (identified by content_id),
for one client. Each row represents a single page's aggregated 90-day
performance snapshot — not a single event or a single day.

In [5]:
!git clone https://github.com/aimanshahid800/flyrank-ml-internship
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 124 (delta 39), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.85 MiB | 16.43 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/flyrank-ml-internship


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("\nOne row = one content page. Example rows:")
df.head()

Shape: (30000, 44)

One row = one content page. Example rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [7]:
# Sketch of the target column
df["is_declining_label"] = df["trend_direction"] == "down"

print(df[["content_id", "trend_direction", "is_declining_label"]].head(10))
print("\nHow many declining pages:", df["is_declining_label"].sum(), "out of", len(df))

             content_id trend_direction  is_declining_label
0  content_304f48230142            down                True
1  content_a1fb4e703a9e            down                True
2  content_9aa793d4d895            down                True
3  content_331d6c4de07b          stable               False
4  content_d99b7a2d90ca            down                True
5  content_d4084a4bc775            down                True
6  content_9a34b442b552            down                True
7  content_a63219c6e95a          stable               False
8  content_5e6c160719bc            down                True
9  content_c27558df2b0c            down                True

How many declining pages: 16262 out of 30000


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

ML beats a fixed rule here because decline is not caused by one single
condition — it depends on many signals interacting together: impressions,
position, freshness, CTR, engagement rate, scroll rate, and word count all
matter differently for different types of pages. A single if-statement can
only check one or two conditions at a time, but a model can weigh dozens
of signals together and learn which combinations actually predict decline.

This is proven by the numbers from this dataset: the fixed baseline rule
scores only 0.240 on Precision@50, while a random forest model scores
0.740 — nearly 3x better at correctly identifying the pages that need
review first.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.